In [1]:
import pandas as pd
import numpy as np




In [2]:
train_df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/train.csv")
test_df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/test.csv")
sub_df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/sample_submission.csv")

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("sub_df shape:", sub_df.shape)




train shape: (49232, 14)
test shape: (5474, 9)
sub_df shape: (2384, 2)


In [3]:
def get_num_unique(train_df, test_df, col):
    all_df = pd.concat([train_df, test_df])
    num_unique_train = len(train_df[col].unique())
    num_unique_test = len(test_df[col].unique())
    num_unique_all = len(all_df[col].unique())
    return num_unique_train, num_unique_test, num_unique_all


def add_count(df, col):
    if isinstance(col, str):
        aggs = df.groupby(col, as_index=True)[col].count().rename(col + "_count")
    else:
        aggs = (
            df.groupby(col, as_index=False)[col[0]]
            .count()
            .rename("_".join(col) + "_count")
        )
    df = df.merge(aggs, on=col, how="inner")
    return df




In [4]:
hypotheses = []




In [5]:
site_prob = train_df.groupby("site_id")["cancer"].mean()
lat_prob = train_df.groupby("laterality")["cancer"].mean()
view_prob = train_df.groupby("view")["cancer"].mean()
overall_mean = train_df["cancer"].mean()

age_rounded = train_df["age"].fillna(-1).astype(int)
age_prob = train_df.groupby(age_rounded)["cancer"].mean()
implant_prob = train_df.groupby("implant")["cancer"].mean()

site_lat_prob = train_df.groupby(["site_id", "laterality"])["cancer"].mean()
site_view_prob = train_df.groupby(["site_id", "view"])["cancer"].mean()
lat_view_prob = train_df.groupby(["laterality", "view"])["cancer"].mean()

patient_lat_prob = train_df.groupby(["patient_id", "laterality"])["cancer"].mean()
patient_prob = train_df.groupby("patient_id")["cancer"].mean()

machine_prob = train_df.groupby("machine_id")["cancer"].mean()

if "BIRADS" in train_df.columns:
    birads_prob = train_df.groupby("BIRADS")["cancer"].mean()
else:
    birads_prob = pd.Series(dtype=float)

if "density" in train_df.columns:
    density_prob = train_df.groupby("density")["cancer"].mean()
else:
    density_prob = pd.Series(dtype=float)

if "biopsy" in train_df.columns:
    biopsy_prob = train_df.groupby("biopsy")["cancer"].mean()
else:
    biopsy_prob = pd.Series(dtype=float)


def estimate_prob(row):
    probs = []
    if row["site_id"] in site_prob.index:
        probs.append(site_prob[row["site_id"]])
    if row["laterality"] in lat_prob.index:
        probs.append(lat_prob[row["laterality"]])
    if row["view"] in view_prob.index:
        probs.append(view_prob[row["view"]])
    age_key = int(row["age"]) if not pd.isna(row["age"]) else -1
    if age_key in age_prob.index:
        probs.append(age_prob[age_key])
    if row["implant"] in implant_prob.index:
        probs.append(implant_prob[row["implant"]])
    if (row["site_id"], row["laterality"]) in site_lat_prob.index:
        probs.append(site_lat_prob[(row["site_id"], row["laterality"])])
    if (row["site_id"], row["view"]) in site_view_prob.index:
        probs.append(site_view_prob[(row["site_id"], row["view"])])
    if (row["laterality"], row["view"]) in lat_view_prob.index:
        probs.append(lat_view_prob[(row["laterality"], row["view"])])
    patient_key = (row["patient_id"], row["laterality"])
    if patient_key in patient_lat_prob.index:
        probs.append(patient_lat_prob[patient_key])
    if row["patient_id"] in patient_prob.index:
        probs.append(patient_prob[row["patient_id"]])
    if row["machine_id"] in machine_prob.index:
        probs.append(machine_prob[row["machine_id"]])
    if "BIRADS" in row and row["BIRADS"] in birads_prob.index:
        probs.append(birads_prob[row["BIRADS"]])
    if "density" in row and row["density"] in density_prob.index:
        probs.append(density_prob[row["density"]])
    if "biopsy" in row and row["biopsy"] in biopsy_prob.index:
        probs.append(biopsy_prob[row["biopsy"]])
    probs.append(overall_mean)
    return np.mean(probs) if probs else overall_mean


test_df["cancer"] = test_df.apply(estimate_prob, axis=1)

patient_map = test_df["patient_id"].map(patient_prob).fillna(overall_mean)
test_df["cancer"] = 0.4 * test_df["cancer"] + 0.6 * patient_map

test_df["cancer"] = test_df["cancer"].clip(0, 1)

submission = test_df.groupby("prediction_id")["cancer"].mean().reset_index()

# Light min‑max scaling to increase prediction contrast (move score toward target)
c = submission["cancer"]
c_min, c_max = c.min(), c.max()
if c_max > c_min:
    c_scaled = (c - c_min) / (c_max - c_min)
else:
    c_scaled = c
# Blend a tiny amount of the overall mean to keep probabilities well‑behaved
submission["cancer"] = 0.95 * c_scaled + 0.05 * overall_mean

submission.to_csv("submission.csv", index=False)
display(submission.head())

,prediction_id,cancer
0,10116_L,0.412910
1,10116_R,0.409984
2,10130_L,0.585214
3,10130_R,0.550404
4,10188_L,0.328955
